# 11. EDA Insights

## Objective

This notebook consolidates the most important findings from the EDA, cleaning, feature engineering, and feature-selection stages.

It is designed to:
- consolidate the key findings from all EDA notebooks
- translate technical observations into business understanding
- identify modeling implications
- define recommendations for the modeling phase


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import pointbiserialr
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RANDOM_STATE, TARGET_COLUMN, TEST_SIZE

raw_data_path = project_root / "data" / "raw" / "telco.csv"
cleaned_data_path = project_root / "data" / "interim" / "telco_churn_cleaned.csv"
engineered_data_path = project_root / "data" / "processed" / "telco_churn_engineered.csv"
feature_selection_notebook_path = project_root / "notebooks" / "10_feature_selection.ipynb"

if not engineered_data_path.exists():
    raise FileNotFoundError(
        "Engineered dataset not found. Run notebook 09_feature_engineering.ipynb first "
        f"to create: {engineered_data_path}"
    )

raw_df = pd.read_csv(raw_data_path) if raw_data_path.exists() else None
cleaned_df = pd.read_csv(cleaned_data_path) if cleaned_data_path.exists() else None
engineered_df = pd.read_csv(engineered_data_path)

df = engineered_df.copy()
if "Churn_bin" not in df.columns:
    raise KeyError("Churn_bin was not found in the engineered dataset.")

target_column = TARGET_COLUMN
id_columns = [column for column in ["customerID"] if column in df.columns]
label_columns = [column for column in ["Churn", "Churn_bin"] if column in df.columns]
feature_columns = [column for column in df.columns if column not in id_columns + label_columns]

X = df[feature_columns].copy()
y = df["Churn_bin"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()
binary_features = [
    column
    for column in numeric_features
    if set(pd.Series(X[column]).dropna().astype(float).unique()).issubset({0.0, 1.0})
]
continuous_numeric_features = [column for column in numeric_features if column not in binary_features]
engineered_feature_markers = (
    "_bin",
    "_ordinal",
    "_group",
    "_profile",
    "_log",
    "_band",
    "_x_",
    "is_",
)
engineered_features = [
    column for column in feature_columns if any(marker in column for marker in engineered_feature_markers)
]
original_features = [column for column in feature_columns if column not in engineered_features]

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Dataset Recap

This section gives a short reminder of the modeling-ready dataset, target definition, feature groups, and current split status.


In [2]:
dataset_recap = pd.DataFrame(
    {
        "metric": [
            "dataset used for insights",
            "dataset shape",
            "target column",
            "problem type",
            "original feature count",
            "engineered feature count",
            "numeric feature count",
            "categorical feature count",
            "train/test split status",
            "train shape",
            "test shape",
        ],
        "value": [
            str(engineered_data_path.relative_to(project_root)),
            f"{df.shape[0]} rows x {df.shape[1]} columns",
            f"{target_column} / Churn_bin",
            "binary classification",
            len(original_features),
            len(engineered_features),
            len(numeric_features),
            len(categorical_features),
            f"virtual split prepared in notebook using TEST_SIZE={TEST_SIZE}",
            f"{X_train.shape[0]} rows x {X_train.shape[1]} features",
            f"{X_test.shape[0]} rows x {X_test.shape[1]} features",
        ],
    }
)

dataset_recap


,metric,value
0,dataset used for insights,data/processed/telco_churn_engineered.csv
1,dataset shape,7043 rows x 43 columns
2,target column,Churn / Churn_bin
3,problem type,binary classification
4,original feature count,22
5,engineered feature count,18
6,numeric feature count,19
7,categorical feature count,21
8,train/test split status,virtual split prepared in notebook using TEST_...
9,train shape,5634 rows x 40 features


## Key Data Quality Findings

This section summarizes the main cleaning findings from earlier notebooks and converts them into a compact action log for modeling.


In [12]:
quality_rows = []

if raw_df is not None:
    totalcharges_blank_mask = raw_df["TotalCharges"].astype(str).str.strip().eq("")
    quality_rows.append(
        {
            "issue": "Blank TotalCharges values in raw data",
            "finding": int(totalcharges_blank_mask.sum()),
            "detail": "Blank values are present in the raw file and should not be treated as random missingness.",
            "action_taken": "Handled during cleaning and converted into numeric TotalCharges in downstream datasets.",
        }
    )
    if "tenure" in raw_df.columns:
        quality_rows.append(
            {
                "issue": "Blank TotalCharges linked to tenure",
                "finding": int(raw_df.loc[totalcharges_blank_mask, "tenure"].eq(0).sum()),
                "detail": "These records are typically new customers with tenure = 0.",
                "action_taken": "Kept records and cleaned TotalCharges consistently instead of dropping them blindly.",
            }
        )
    quality_rows.append(
        {
            "issue": "Duplicate rows in raw data",
            "finding": int(raw_df.duplicated().sum()),
            "detail": "Checks whether exact duplicates exist in the source dataset.",
            "action_taken": "No large duplicate issue expected for this dataset; continue to keep customerID out of modeling.",
        }
    )

quality_rows.append(
    {
        "issue": "Missing values in engineered dataset",
        "finding": int(df.isna().sum().sum()),
        "detail": "The processed modeling dataset should be numerically stable and ready for downstream pipelines.",
        "action_taken": "Review any remaining nulls before fitting final models.",
    }
)
quality_rows.append(
    {
        "issue": "Duplicate rows in engineered dataset",
        "finding": int(df.duplicated().sum()),
        "detail": "A quick check on the final engineered table.",
        "action_taken": "Duplicate rows should not be passed into modeling if discovered later.",
    }
)

skewness_df = (
    df[continuous_numeric_features]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
    .rename("skewness")
    .reset_index()
    .rename(columns={"index": "feature"})
)
strong_skew = skewness_df.loc[skewness_df["skewness"].abs() >= 1].copy()

quality_rows.append(
    {
        "issue": "Strong skew in continuous numeric variables",
        "finding": int(strong_skew.shape[0]),
        "detail": ", ".join(strong_skew["feature"].head(6).tolist()) if not strong_skew.empty else "No strongly skewed continuous features detected.",
        "action_taken": "Use transformed versions such as TotalCharges_log when they improve stability and interpretability.",
    }
)

quality_summary_df = pd.DataFrame(quality_rows)
quality_summary_df


,issue,finding,detail,action_taken
0,Blank TotalCharges values in raw data,11,Blank values are present in the raw file and s...,Handled during cleaning and converted into num...
1,Blank TotalCharges linked to tenure,11,These records are typically new customers with...,Kept records and cleaned TotalCharges consiste...
2,Duplicate rows in raw data,0,Checks whether exact duplicates exist in the s...,No large duplicate issue expected for this dat...
3,Missing values in engineered dataset,0,The processed modeling dataset should be numer...,Review any remaining nulls before fitting fina...
4,Duplicate rows in engineered dataset,0,A quick check on the final engineered table.,Duplicate rows should not be passed into model...
5,Strong skew in continuous numeric variables,0,No strongly skewed continuous features detected.,Use transformed versions such as TotalCharges_...


The only cleaning issue of note was the 11 blank `TotalCharges` records, all tied to `tenure = 0` new customers and handled consistently during preprocessing. After that cleanup, the engineered dataset is model-ready with zero missing values and no remaining data-quality concerns that materially affect the next phase.


## Univariate Insights

The goal here is not to recreate every plot, but to extract the most important single-variable patterns that matter for churn understanding and modeling.


In [13]:
class_balance_df = (
    df["Churn"]
    .value_counts(dropna=False)
    .rename_axis("Churn")
    .reset_index(name="count")
)
class_balance_df["share_pct"] = (class_balance_df["count"] / len(df) * 100).round(2)

numeric_distribution_df = df[continuous_numeric_features].describe().T.reset_index().rename(columns={"index": "feature"})
numeric_distribution_df["skewness"] = numeric_distribution_df["feature"].map(
    dict(zip(skewness_df["feature"], skewness_df["skewness"]))
).round(3)

categorical_cardinality_df = pd.DataFrame(
    {
        "feature": categorical_features,
        "unique_categories": [df[column].nunique(dropna=False) for column in categorical_features],
        "top_category": [df[column].mode(dropna=False).iloc[0] for column in categorical_features],
        "top_category_share_pct": [round(df[column].value_counts(normalize=True, dropna=False).iloc[0] * 100, 2) for column in categorical_features],
    }
).sort_values(["unique_categories", "top_category_share_pct"], ascending=[False, False])

print("Class balance")
display(class_balance_df)
print("\nContinuous feature summary")
display(numeric_distribution_df[["feature", "mean", "std", "min", "25%", "50%", "75%", "max", "skewness"]].sort_values("skewness", ascending=False))
print("\nCategorical dominance snapshot")
display(categorical_cardinality_df.head(12))


Class balance


,Churn,count,share_pct
0,No,5174,73.46
1,Yes,1869,26.54



Continuous feature summary


,feature,mean,std,min,25%,50%,75%,max,skewness
2,TotalCharges,2279.734304,2266.794470,0.000,398.550000,1394.550000,3786.600000,8684.800000,0.963
5,tenure_x_MonthlyCharges,2279.581350,2264.729447,0.000,394.000000,1393.600000,3786.100000,8550.000000,0.961
4,Contract_ordinal,0.690473,0.833755,0.000,0.000000,0.000000,1.000000,2.000000,0.631
7,service_count,2.459747,2.045539,0.000,1.000000,2.000000,4.000000,7.000000,0.415
0,tenure,32.371149,24.559481,0.000,9.000000,29.000000,55.000000,72.000000,0.240
6,avg_monthly_spend,64.762906,30.189796,13.775,35.935156,70.337500,90.174158,121.400000,-0.210
1,MonthlyCharges,64.761692,30.090047,18.250,35.500000,70.350000,89.850000,118.750000,-0.221
3,TotalCharges_log,6.932543,1.569371,0.000,5.990339,7.241044,8.239488,9.069445,-0.824



Categorical dominance snapshot


,feature,unique_categories,top_category,top_category_share_pct
18,contract_payment_profile,12,Month-to-month__Electronic check,26.27
20,internet_onlinesecurity_profile,5,Fiber optic__No,32.05
19,internet_techsupport_profile,5,Fiber optic__No,31.66
14,PaymentMethod,4,Electronic check,33.58
15,tenure_band,4,long_term,31.79
12,Contract,3,Month-to-month,55.02
6,OnlineSecurity,3,No,49.67
9,TechSupport,3,No,49.31
4,MultipleLines,3,No,48.13
5,InternetService,3,Fiber optic,43.96


**Observed univariate takeaways:**

- Churn is moderately imbalanced: 26.54% of customers churned and 73.46% stayed, so accuracy alone would be misleading during model evaluation.
- The main continuous business variables are broad rather than tightly clustered. `MonthlyCharges` averages 64.76, `TotalCharges` averages 2279.73, and `tenure` centers around 29 months with a wide spread up to 72 months.
- Skew is present but not extreme in most modeling variables. `TotalCharges` and `tenure_x_MonthlyCharges` are the most right-skewed continuous fields at about 0.96, while `TotalCharges_log` reduces that shape distortion and is more modeling-friendly.
- Several engineered categorical fields are high-cardinality compared with the original dataset, especially `contract_payment_profile` with 12 categories and the internet support profile fields with 5 categories each. These are useful for insight generation, but they need to be controlled carefully to avoid sparse one-hot expansion.
- Month-to-month contract is already the dominant customer category at 55.02% of the dataset, and `Electronic check` is the most common payment method at 33.58%, which makes both features important to inspect closely in churn analysis.


## Bivariate Insights With Target

This is one of the most important sections. It summarizes which variables show the clearest relationship with churn and which patterns appear strongest from both statistical and business viewpoints.


In [5]:
numeric_target_rows = []
for column in continuous_numeric_features + binary_features:
    series = df[column].astype(float)
    corr_value = series.corr(y)
    point_r, point_p = pointbiserialr(y, series)
    numeric_target_rows.append(
        {
            "feature": column,
            "pearson_with_target": round(float(corr_value), 4),
            "abs_pearson_with_target": round(abs(float(corr_value)), 4),
            "pointbiserial_r": round(float(point_r), 4),
            "pointbiserial_p": float(point_p),
        }
    )

numeric_target_df = pd.DataFrame(numeric_target_rows).sort_values(
    "abs_pearson_with_target", ascending=False
).reset_index(drop=True)

categorical_target_rows = []
for column in categorical_features:
    grouped = (
        df.groupby(column, dropna=False)["Churn_bin"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={column: "category_value", "mean": "churn_rate", "count": "segment_count"})
    )
    grouped["feature"] = column
    grouped["overall_gap"] = (grouped["churn_rate"] - df["Churn_bin"].mean()).abs()
    categorical_target_rows.append(grouped)

categorical_target_df = pd.concat(categorical_target_rows, ignore_index=True)
categorical_target_df["churn_rate_pct"] = (categorical_target_df["churn_rate"] * 100).round(2)
categorical_target_df = categorical_target_df.sort_values(
    ["overall_gap", "segment_count"], ascending=[False, False]
).reset_index(drop=True)

print("Top numeric and binary relationships with churn")
display(numeric_target_df.head(15))
print("\nTop categorical segments by churn-rate gap")
display(categorical_target_df[["feature", "category_value", "segment_count", "churn_rate_pct", "overall_gap"]].head(20))


Top numeric and binary relationships with churn


,feature,pearson_with_target,abs_pearson_with_target,pointbiserial_r,pointbiserial_p
0,is_month_to_month,0.4051,0.4051,0.4051,1.991701e-276
1,Contract_ordinal,-0.3967,0.3967,-0.3967,3.666675e-264
2,tenure,-0.3522,0.3522,-0.3522,7.999058e-205
3,is_new_customer,0.3177,0.3177,0.3177,6.867572e-165
4,TotalCharges_log,-0.2340,0.2340,-0.2340,3.437483e-88
5,is_auto_payment,-0.2099,0.2099,-0.2099,5.789070e-71
6,tenure_x_MonthlyCharges,-0.1985,0.1985,-0.1985,1.612238e-63
7,TotalCharges,-0.1983,0.1983,-0.1983,2.127212e-63
8,MonthlyCharges,0.1934,0.1934,0.1934,2.706646e-60
9,avg_monthly_spend,0.1925,0.1925,0.1925,8.716816e-60



Top categorical segments by churn-rate gap


,feature,category_value,segment_count,churn_rate_pct,overall_gap
0,contract_payment_profile,Month-to-month__Electronic check,1850,53.73,0.271927
1,contract_payment_profile,Two year__Mailed check,382,0.79,0.257516
2,contract_payment_profile,Two year__Credit card (automatic),581,2.24,0.242995
3,Contract,Two year,1695,2.83,0.237051
4,contract_payment_profile,Two year__Bank transfer (automatic),564,3.37,0.231682
5,internet_techsupport_profile,Fiber optic__No,2230,49.37,0.228352
6,internet_onlinesecurity_profile,Fiber optic__No,2257,49.36,0.228206
7,tenure_band,new,2186,47.44,0.209013
8,contract_payment_profile,One year__Mailed check,337,6.82,0.197121
9,InternetService,No,1526,7.40,0.191320


**Observed bivariate takeaways:**

- Contract and lifecycle variables are the clearest churn signals in the dataset. The strongest numeric relationships with churn are `is_month_to_month` (r = 0.4051), `Contract_ordinal` (r = -0.3967), `tenure` (r = -0.3522), and `is_new_customer` (r = 0.3177).
- Spend-related variables also matter, but less strongly than contract and tenure. `TotalCharges_log` has a correlation of -0.2340 with churn, while `MonthlyCharges` and `avg_monthly_spend` are positively associated with churn at about 0.19.
- Payment behavior clearly separates risk segments. The `Month-to-month__Electronic check` profile has a churn rate of 53.73%, while stable long-commitment payment combinations such as `Two year__Mailed check` and `Two year__Credit card (automatic)` fall to 0.79% and 2.24%.
- Support and internet-service patterns are practically important, not just statistically significant. `Fiber optic__No` in both support-profile tables shows churn rates around 49.36% to 49.37%, compared with only 7.40% for no-internet-service groups.
- Newer customers are especially vulnerable: the `new` tenure band shows a 47.44% churn rate, confirming that early lifecycle retention should be a top business priority.


## Multivariate Insights

This section explains combined patterns instead of isolated variables. It highlights redundancy, interaction structure, and churn-risk profiles that become clearer when variables are read together.


In [6]:
numeric_corr_matrix = df[continuous_numeric_features + binary_features].corr(numeric_only=True)

high_corr_pairs = []
for left_index, left_feature in enumerate(numeric_corr_matrix.columns):
    for right_feature in numeric_corr_matrix.columns[left_index + 1:]:
        corr_value = numeric_corr_matrix.loc[left_feature, right_feature]
        if abs(corr_value) >= 0.7:
            high_corr_pairs.append(
                {
                    "feature_1": left_feature,
                    "feature_2": right_feature,
                    "correlation": round(float(corr_value), 4),
                    "abs_correlation": round(abs(float(corr_value)), 4),
                }
            )

high_corr_pairs_df = pd.DataFrame(high_corr_pairs).sort_values("abs_correlation", ascending=False)

combined_risk_profiles = (
    df.groupby(["Contract", "payment_method_group", "has_support_services"], dropna=False)["Churn_bin"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "churn_rate", "count": "segment_count"})
)
combined_risk_profiles = combined_risk_profiles.loc[combined_risk_profiles["segment_count"] >= 50].copy()
combined_risk_profiles["churn_rate_pct"] = (combined_risk_profiles["churn_rate"] * 100).round(2)
combined_risk_profiles = combined_risk_profiles.sort_values(["churn_rate", "segment_count"], ascending=[False, False])

print("High-correlation pairs (|r| >= 0.7)")
display(high_corr_pairs_df.head(20) if not high_corr_pairs_df.empty else pd.DataFrame(columns=["feature_1", "feature_2", "correlation", "abs_correlation"]))
print("\nCombined churn-risk profiles")
display(combined_risk_profiles.head(15))


High-correlation pairs (|r| >= 0.7)


,feature_1,feature_2,correlation,abs_correlation
15,SeniorCitizen,SeniorCitizen_bin,1.0000,1.0000
7,TotalCharges,tenure_x_MonthlyCharges,0.9996,0.9996
4,MonthlyCharges,avg_monthly_spend,0.9962,0.9962
11,Contract_ordinal,is_month_to_month,-0.9160,0.9160
1,tenure,TotalCharges_log,0.8291,0.8291
2,tenure,tenure_x_MonthlyCharges,0.8266,0.8266
0,tenure,TotalCharges,0.8262,0.8262
6,TotalCharges,TotalCharges_log,0.8258,0.8258
9,TotalCharges_log,tenure_x_MonthlyCharges,0.8258,0.8258
10,TotalCharges_log,is_new_customer,-0.7960,0.7960



Combined churn-risk profiles


,Contract,payment_method_group,has_support_services,churn_rate,segment_count,churn_rate_pct
2,Month-to-month,electronic_check,0,0.619289,985,61.93
3,Month-to-month,electronic_check,1,0.443931,865,44.39
0,Month-to-month,auto_payment,0,0.366534,502,36.65
4,Month-to-month,manual_check,0,0.338409,591,33.84
1,Month-to-month,auto_payment,1,0.309524,630,30.95
5,Month-to-month,manual_check,1,0.271523,302,27.15
8,One year,electronic_check,0,0.187500,80,18.75
9,One year,electronic_check,1,0.183521,267,18.35
7,One year,auto_payment,1,0.114695,558,11.47
11,One year,manual_check,1,0.103226,155,10.32


**Observed multivariate takeaways:**

- Several feature families contain very strong redundancy. `TotalCharges` is almost interchangeable with `tenure_x_MonthlyCharges` (r = 0.9996), `MonthlyCharges` is nearly identical to `avg_monthly_spend` (r = 0.9962), and `Contract_ordinal` overlaps heavily with `is_month_to_month` (r = -0.9160).
- Lifecycle variables also overlap strongly with churn-timing flags. `tenure` correlates 0.8291 with `TotalCharges_log` and -0.7367 with `is_new_customer`, which means the final model should not keep every lifecycle representation by default.
- Combined customer profiles are more informative than isolated features. Month-to-month customers paying by electronic check without support services have the highest observed churn rate at 61.93%, while the same contract group with support still churns at 44.39%.
- Contract commitment changes the whole risk picture. Two-year customers on auto-payment show churn rates as low as 1.03% to 3.71%, which is dramatically lower than the highest-risk month-to-month segments.
- These results support selective use of engineered interaction features: they add business clarity, but many of them sit on top of already-strong base features and should be pruned carefully for linear models.


## Feature Engineering Insights

This section summarizes which engineering choices were useful, which ones improved interpretability, and which ones may need restraint during modeling.


In [7]:
engineering_summary_df = pd.DataFrame(
    {
        "feature_group": [
            "log-transformed fields",
            "binary encodings",
            "ordinal encodings",
            "grouped categorical fields",
            "interaction / profile fields",
            "customer-behavior flags",
        ],
        "matching_features": [
            ", ".join([column for column in engineered_features if column.endswith("_log")]) or "None detected",
            ", ".join([column for column in engineered_features if column.endswith("_bin")]) or "None detected",
            ", ".join([column for column in engineered_features if column.endswith("_ordinal")]) or "None detected",
            ", ".join([column for column in engineered_features if column.endswith("_group") or column.endswith("_band")]) or "None detected",
            ", ".join([column for column in engineered_features if "_profile" in column or "_x_" in column]) or "None detected",
            ", ".join([column for column in engineered_features if column.startswith("is_") or column in {"service_count", "has_support_services", "avg_monthly_spend"}]) or "None detected",
        ],
        "modeling_note": [
            "Use transformed versions when they materially reduce skew or stabilize scale.",
            "Useful for compact linear-model inputs, but avoid keeping raw and encoded duplicates together.",
            "Helpful when there is a natural order, such as contract duration.",
            "Improves interpretability by reducing sparse category fragmentation.",
            "Captures business behavior but can introduce overlapping signal with base columns.",
            "Often strong churn proxies, but should be checked for leakage-style shortcuts and redundancy.",
        ],
    }
)

engineering_summary_df


,feature_group,matching_features,modeling_note
0,log-transformed fields,TotalCharges_log,Use transformed versions when they materially ...
1,binary encodings,"gender_bin, partner_bin, dependents_bin, phone...","Useful for compact linear-model inputs, but av..."
2,ordinal encodings,Contract_ordinal,"Helpful when there is a natural order, such as..."
3,grouped categorical fields,"tenure_band, payment_method_group, internet_se...",Improves interpretability by reducing sparse c...
4,interaction / profile fields,"tenure_x_MonthlyCharges, contract_payment_prof...",Captures business behavior but can introduce o...
5,customer-behavior flags,"is_new_customer, is_month_to_month, is_auto_pa...","Often strong churn proxies, but should be chec..."


Customer-behavior flags and contract encodings are the most analytically useful engineered groups because they compress churn-relevant lifecycle and commitment signals into features that are both interpretable and model-ready. The interaction and profile fields add valuable business context, but they need the most restraint during modeling because they can quickly introduce sparse columns and overlap heavily with already-strong base features.


## Outlier And Scaling Implications

This section clarifies whether observed outliers should be capped, transformed, or left intact, and how scaling should be handled in the modeling phase.


In [14]:
outlier_rows = []
for column in continuous_numeric_features:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((df[column] < lower_bound) | (df[column] > upper_bound)).sum())
    if column == "TotalCharges_log":
        recommended_action = (
            "The 11 detected outliers align with the tenure = 0 new-customer records already documented during cleaning. "
            "They are expected edge cases and should be retained rather than capped or removed."
        )
    elif outlier_count == 0:
        recommended_action = (
            "No material outlier issue is detected under the IQR rule; keep the feature as-is and handle scaling only inside the modeling pipeline if needed."
        )
    else:
        recommended_action = (
            "Review the distribution during modeling, but avoid automatic capping unless later diagnostics suggest these values behave like data errors rather than valid customer behavior."
        )
    outlier_rows.append(
        {
            "feature": column,
            "iqr_outlier_count": outlier_count,
            "iqr_outlier_share_pct": round(outlier_count / len(df) * 100, 2),
            "recommended_action": recommended_action,
        }
    )

outlier_summary_df = pd.DataFrame(outlier_rows).sort_values("iqr_outlier_share_pct", ascending=False)

scaling_guidance_df = pd.DataFrame(
    {
        "model_family": [
            "Logistic Regression",
            "Linear SVM / distance-based methods",
            "Tree ensembles (Random Forest, XGBoost, LightGBM)",
        ],
        "scaling_needed": ["Yes", "Yes", "Usually no"],
        "implementation_note": [
            "Apply scaling inside a leakage-safe sklearn pipeline.",
            "Scale numeric features inside the training pipeline only.",
            "Keep preprocessing simpler, but still encode categories consistently.",
        ],
    }
)

print("Outlier summary")
display(outlier_summary_df)
print("\nScaling guidance")
display(scaling_guidance_df)


Outlier summary


,feature,iqr_outlier_count,iqr_outlier_share_pct,recommended_action
3,TotalCharges_log,11,0.16,The 11 detected outliers align with the tenure...
0,tenure,0,0.00,No material outlier issue is detected under th...
1,MonthlyCharges,0,0.00,No material outlier issue is detected under th...
2,TotalCharges,0,0.00,No material outlier issue is detected under th...
4,Contract_ordinal,0,0.00,No material outlier issue is detected under th...
5,tenure_x_MonthlyCharges,0,0.00,No material outlier issue is detected under th...
6,avg_monthly_spend,0,0.00,No material outlier issue is detected under th...
7,service_count,0,0.00,No material outlier issue is detected under th...



Scaling guidance


,model_family,scaling_needed,implementation_note
0,Logistic Regression,Yes,Apply scaling inside a leakage-safe sklearn pi...
1,Linear SVM / distance-based methods,Yes,Scale numeric features inside the training pip...
2,"Tree ensembles (Random Forest, XGBoost, LightGBM)",Usually no,"Keep preprocessing simpler, but still encode c..."


## Feature Selection Insights

This section bridges notebook 10 into the modeling phase by summarizing the strongest candidate features, redundant families, and feature-handling cautions.

Contract-related features need explicit redundancy control before modeling. `Contract`, `Contract_ordinal`, and `is_month_to_month` all describe the same customer-commitment signal, and the multivariate review showed especially strong overlap between `Contract_ordinal` and `is_month_to_month` (r = -0.9160). For linear and interpretation-focused models such as Logistic Regression, keep `Contract_ordinal` as the main contract feature and drop both raw `Contract` and `is_month_to_month` to reduce multicollinearity and keep coefficients easier to explain. For tree-based models, test either one-hot encoded `Contract` or the simpler `is_month_to_month` flag, but avoid carrying all three versions into the same final feature set unless you are doing a deliberate comparison experiment.


In [9]:
feature_selection_snapshot = {
    "kept_feature_count": None,
    "review_feature_count": None,
    "dropped_feature_count": None,
    "top_keep_features": [],
}

if feature_selection_notebook_path.exists():
    notebook_json = json.loads(feature_selection_notebook_path.read_text())
    for cell in notebook_json.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source_text = "".join(cell.get("source", []))
        if "final_selection_summary" not in source_text:
            continue
        for output in cell.get("outputs", []):
            output_text = ""
            if "text" in output:
                output_text = "".join(output["text"])
            elif "data" in output and "text/plain" in output["data"]:
                output_text = "".join(output["data"]["text/plain"])
            if "kept_feature_count" in output_text and "top_keep_features" in output_text:
                parsed = json.loads(
                    output_text.replace("'", '"').replace("None", "null")
                )
                feature_selection_snapshot.update(parsed)
                break

feature_selection_summary_df = pd.DataFrame(
    {
        "metric": list(feature_selection_snapshot.keys()),
        "value": [
            ", ".join(value[:15]) if isinstance(value, list) else value
            for value in feature_selection_snapshot.values()
        ],
    }
)

likely_high_value_families_df = pd.DataFrame(
    {
        "feature_family": [
            "contract commitment",
            "customer lifecycle",
            "spend and billing",
            "support and security services",
            "payment behavior",
            "engineered interaction flags",
        ],
        "representative_features": [
            "Contract, Contract_ordinal, is_month_to_month",
            "tenure, tenure_band, is_new_customer",
            "MonthlyCharges, TotalCharges, TotalCharges_log, avg_monthly_spend",
            "OnlineSecurity, TechSupport, has_support_services, profile features",
            "PaymentMethod, payment_method_group, is_auto_payment",
            "tenure_x_MonthlyCharges, contract_payment_profile, internet_*_profile",
        ],
        "modeling_note": [
            "Usually one of the strongest churn families; watch for redundant encodings.",
            "Important for early-churn detection and retention timing.",
            "Potentially strong but partially redundant with tenure-driven lifecycle signal.",
            "Often highly interpretable and useful for targeting retention offers.",
            "Can reflect friction and customer behavior patterns.",
            "Retain selectively; some interaction fields may be too sparse for simple baselines.",
        ],
    }
)

print("Feature-selection snapshot from notebook 10")
display(feature_selection_summary_df)
print("\nFeature families to carry into modeling")
display(likely_high_value_families_df)


Feature-selection snapshot from notebook 10


,metric,value
0,kept_feature_count,39
1,review_feature_count,27
2,dropped_feature_count,20
3,top_keep_features,"Contract_ordinal, internet_service_group_fiber..."



Feature families to carry into modeling


,feature_family,representative_features,modeling_note
0,contract commitment,"Contract, Contract_ordinal, is_month_to_month",Usually one of the strongest churn families; w...
1,customer lifecycle,"tenure, tenure_band, is_new_customer",Important for early-churn detection and retent...
2,spend and billing,"MonthlyCharges, TotalCharges, TotalCharges_log...",Potentially strong but partially redundant wit...
3,support and security services,"OnlineSecurity, TechSupport, has_support_servi...",Often highly interpretable and useful for targ...
4,payment behavior,"PaymentMethod, payment_method_group, is_auto_p...",Can reflect friction and customer behavior pat...
5,engineered interaction flags,"tenure_x_MonthlyCharges, contract_payment_prof...",Retain selectively; some interaction fields ma...


The current feature snapshot is well-balanced: 39 features are in `keep`, 27 remain in `review`, and 20 are in `drop`, which narrows the engineered space without over-pruning useful signal. The `review` group is now the main modeling decision area, because some of those features will be more appropriate for tree-based models than for linear baselines with stricter redundancy control.


## Business Insights

This section should read like an executive-ready summary. The purpose is to convert technical findings into practical churn meaning and retention implications.


In [10]:
manager_view_df = (
    df.groupby(["Contract", "tenure_band", "has_support_services"], dropna=False)["Churn_bin"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "churn_rate", "count": "segment_count"})
)
manager_view_df = manager_view_df.loc[manager_view_df["segment_count"] >= 40].copy()
manager_view_df["churn_rate_pct"] = (manager_view_df["churn_rate"] * 100).round(2)
manager_view_df = manager_view_df.sort_values(["churn_rate", "segment_count"], ascending=[False, False])

contract_payment_high_risk = combined_risk_profiles.iloc[0]
contract_payment_with_support = combined_risk_profiles.iloc[1]
contract_payment_low_risk = combined_risk_profiles.loc[
    (combined_risk_profiles["Contract"] == "Two year")
    & (combined_risk_profiles["payment_method_group"] == "auto_payment")
    & (combined_risk_profiles["has_support_services"] == 0)
].iloc[0]
new_customer_high_risk = manager_view_df.loc[
    (manager_view_df["Contract"] == "Month-to-month")
    & (manager_view_df["tenure_band"] == "new")
    & (manager_view_df["has_support_services"] == 0)
].iloc[0]

business_takeaways_df = pd.DataFrame(
    {
        "business_theme": [
            "Contract structure",
            "Early lifecycle churn",
            "Value-added support services",
            "Payment behavior",
            "Retention targeting",
        ],
        "implication": [
            f"Month-to-month customers paying by electronic check without support services churn at {contract_payment_high_risk['churn_rate_pct']:.2f}% across {int(contract_payment_high_risk['segment_count'])} customers, confirming that short contract commitment is the clearest retention risk.",
            f"New month-to-month customers without support services churn at {new_customer_high_risk['churn_rate_pct']:.2f}% across {int(new_customer_high_risk['segment_count'])} customers, showing that the earliest lifecycle stage needs proactive retention outreach.",
            f"Month-to-month customers paying by electronic check still churn at {contract_payment_with_support['churn_rate_pct']:.2f}% across {int(contract_payment_with_support['segment_count'])} customers even when support services are present, so support helps but does not fully offset high-risk contract and payment behavior.",
            f"Two-year customers on auto-payment without support services churn at only {contract_payment_low_risk['churn_rate_pct']:.2f}% across {int(contract_payment_low_risk['segment_count'])} customers, showing that payment stability and long commitment strongly align with retention.",
            f"The most actionable near-term retention pools are the {int(contract_payment_high_risk['segment_count'])} month-to-month electronic-check customers without support services and the {int(new_customer_high_risk['segment_count'])} new month-to-month customers without support services, both of which exceed 50% churn risk.",
        ],
    }
)

print("High-risk business segments")
display(manager_view_df.head(15))
print("\nBusiness interpretation summary")
display(business_takeaways_df)


High-risk business segments


,Contract,tenure_band,has_support_services,churn_rate,segment_count,churn_rate_pct
6,Month-to-month,new,0,0.538462,1365,53.85
7,Month-to-month,new,1,0.459459,629,45.95
0,Month-to-month,early,0,0.380000,350,38.00
1,Month-to-month,early,1,0.374677,387,37.47
2,Month-to-month,long_term,0,0.359375,64,35.94
4,Month-to-month,mid,0,0.344482,299,34.45
5,Month-to-month,mid,1,0.320080,503,32.01
3,Month-to-month,long_term,1,0.237410,278,23.74
11,One year,long_term,1,0.137525,509,13.75
9,One year,early,1,0.127907,86,12.79



Business interpretation summary


,business_theme,implication
0,Contract structure,Short-commitment customers usually carry the h...
1,Early lifecycle churn,Newer customers tend to be more fragile and ne...
2,Value-added support services,Security and support-related services may act ...
3,Payment behavior,"Payment patterns can reflect friction, conveni..."
4,Retention targeting,Retention programs should focus on short-tenur...


The business picture is now very clear: the highest-risk segments are concentrated in month-to-month customers, especially newer customers and electronic-check customers without support services, while the lowest-risk segments are anchored by two-year contracts and auto-payment behavior. This spread between 61.93% churn in the most exposed segment and 1.03% in the most stable segment gives the modeling phase a strong signal target and a clear business story to preserve.


## Modeling Recommendations

This is the bridge to the next notebook. It should clearly define how the project should move from insights into model training, validation, and comparison.


In [11]:
modeling_recommendations_df = pd.DataFrame(
    {
        "area": [
            "baseline models",
            "advanced comparisons",
            "validation strategy",
            "class imbalance handling",
            "categorical handling",
            "scaling",
            "evaluation metrics",
            "feature rules",
        ],
        "recommendation": [
            "Start with Logistic Regression as the first benchmark.",
            "Compare with Random Forest, XGBoost, or LightGBM after the baseline is stable.",
            "Use stratified train/test split and stratified cross-validation.",
            "Track class imbalance explicitly; consider class_weight, threshold tuning, and PR-aware evaluation.",
            "Use one-hot encoding for nominal variables inside the pipeline and avoid duplicate raw/encoded representations.",
            "Scale only inside the modeling pipeline for linear or distance-based models.",
            "Monitor ROC-AUC, PR-AUC, F1, recall, precision, and the confusion matrix.",
            "Exclude customerID, review redundant engineered pairs, and guard against leakage-like shortcuts.",
        ],
    }
)

modeling_recommendations_df


,area,recommendation
0,baseline models,Start with Logistic Regression as the first be...
1,advanced comparisons,"Compare with Random Forest, XGBoost, or LightG..."
2,validation strategy,Use stratified train/test split and stratified...
3,class imbalance handling,Track class imbalance explicitly; consider cla...
4,categorical handling,Use one-hot encoding for nominal variables ins...
5,scaling,Scale only inside the modeling pipeline for li...
6,evaluation metrics,"Monitor ROC-AUC, PR-AUC, F1, recall, precision..."
7,feature rules,"Exclude customerID, review redundant engineere..."


## Final EDA Conclusion

The dataset is now in a strong position for modeling. Cleaning decisions have been documented, the main churn drivers are visible, engineered features have improved the signal space, and feature-selection work has narrowed the candidate set into more defensible inputs.

The next phase should move into leakage-safe modeling with a baseline classifier, a small set of stronger comparison models, and evaluation that emphasizes churn-focused metrics rather than raw accuracy alone.
